In [ ]:
import json
from pathlib import Path

import pandas as pd
import polars as pl
import yaml
import numpy as np
import structlog
from create_id_lookup import collect_ids, create_lookup, create_partitions
from common import daterange

with open("../config.budapest.yaml", "r") as fp:
    config = yaml.safe_load(fp)

days = []
for i in config["days"]:
    days += list(daterange(i["start"], i["end"]))

log = structlog.get_logger()

In [3]:
OUTPUT = f"{config['dirs']['output']}/{config['city']}/id_lookup"

In [ ]:
pl.DataFrame([["a", "b", "c"]], schema=["id_with_type"]).with_row_index(
    "device_id", offset=1
)

device_id,id_with_type
u32,str
1,"""a"""
2,"""b"""
3,"""c"""


In [10]:
try:
    with open(f"{OUTPUT}/id_lookup.json", "r") as fp:
        lookup = json.load(fp)
except FileNotFoundError:
    # runs 30 minutes for Budapest data
    ids = collect_ids(
        config["dirs"]["raw_source"],
        id_column=config["id_column"],
        id_type_column=config["id_type_column"],
    )
    log.info(f"number unique devices: {len(np.unique(ids))}")

    lookup = create_lookup(ids)

    Path(OUTPUT).mkdir(parents=True, exist_ok=True)
    with open(f"{OUTPUT}/id_lookup.json", "w") as fp:
        json.dump(lookup, fp, indent=1)

2026-06-24 13:46:49 [info     ] number unique devices: 1652702


## create partitions from original IDs

In [11]:
partition = create_partitions(lookup)

with open(f"{OUTPUT}/partition.json", "w") as fp:
    json.dump(partition, fp, indent=1)

### descriptives

In [12]:
partition_distribution = (
    pd.DataFrame({"partition": partition.values(), "device_id": partition.keys()})
    .groupby("partition")["device_id"]
    .count()
    .reset_index()
)
partition_distribution["device_id"].describe()

count     256.000000
mean     6455.867188
std       114.608857
min      6154.000000
25%      6383.750000
50%      6459.500000
75%      6537.250000
max      6766.000000
Name: device_id, dtype: float64